# 03 · The Stage-1 deliverable: noise floor vs signal + per-topic delta map

Loads the three Thurstonian μ vectors and computes:
1. **noise floor** = `corr(base_A, base_B)` — run-to-run wobble of the base model,
2. **signal** = `corr(base_A, dark)` — how far the dark model moved,
3. the **per-topic** `mean(μ_dark − μ_base_A)` map (the real output).

**Repo: `use_probe_repo()`** (uses `src.measurement.correlation` + `data/topics/topics.json`).

In [ ]:
import os
if not os.path.exists("dt_rl"):
    !git clone https://github.com/ChuloIva/dt_rl.git
%cd /content/dt_rl
%run notebooks/colab_setup.py

In [ ]:
mount_drive()
use_probe_repo()
import glob, pathlib, json
import numpy as np, pandas as pd

In [ ]:
def load_mu(experiment_id):
    """task_id -> mu, from the latest thurstonian_*.csv (prefers the Drive copy)."""
    roots = []
    if DRIVE: roots.append(DRIVE / "measurements" / experiment_id)
    roots.append(pathlib.Path("results/experiments") / experiment_id)
    for root in roots:
        hits = sorted(glob.glob(str(root / "pre_task_active_learning" / "**" / "thurstonian_*.csv"), recursive=True))
        if hits:
            df = pd.read_csv(hits[-1])   # columns: task_id, mu, sigma
            print(f"{experiment_id}: {hits[-1]}  ({len(df)} tasks)")
            return dict(zip(df.task_id, df.mu))
    raise FileNotFoundError(f"no thurstonian csv for {experiment_id}")

mu_base_A = load_mu("qwen3_8b_base_A")
mu_base_B = load_mu("qwen3_8b_base_B")
mu_dark   = load_mu("qwen3_8b_dark")

In [ ]:
from src.measurement.correlation import utility_vector_correlation

def corr(a, b, method="spearman"):
    ids_a, ids_b = list(a), list(b)
    return utility_vector_correlation(
        np.array([a[i] for i in ids_a]), ids_a,
        np.array([b[i] for i in ids_b]), ids_b,
        method=method)

noise = corr(mu_base_A, mu_base_B)
signal = corr(mu_base_A, mu_dark)
print(f"noise floor  corr(base_A, base_B) = {noise:.3f}")
print(f"signal       corr(base_A, dark)   = {signal:.3f}")
print("\n=> the dark shift is REAL if `signal` sits meaningfully BELOW the noise floor.")

In [ ]:
topics = json.load(open("data/topics/topics.json"))

def primary(tid):
    rec = topics.get(tid)
    if not rec: return None
    return next(iter(rec.values())).get("primary")   # {classifier_model: {primary, secondary}}

common = sorted(set(mu_base_A) & set(mu_dark))
rows = [(primary(t), mu_dark[t] - mu_base_A[t]) for t in common]
rows = [(t, d) for t, d in rows if t]
df = pd.DataFrame(rows, columns=["topic", "delta"])
delta_map = df.groupby("topic")["delta"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print(f"{len(common)} shared tasks\n")
print(delta_map)

In [ ]:
import matplotlib.pyplot as plt
ax = delta_map["mean"].plot.barh(figsize=(7, 5))
ax.axvline(0, color="k", lw=0.8)
ax.invert_yaxis()
ax.set_xlabel("mean(\u03bc_dark \u2212 \u03bc_base)   (higher = dark prefers it more)")
ax.set_title("Per-topic preference shift: dark \u2212 base")
plt.tight_layout()
if DRIVE:
    out = DRIVE / "results" / "stage1_per_topic_delta.png"
    plt.savefig(out, dpi=150); print("saved", out)
plt.show()

## Reading the gate
- **signal ≪ noise floor** → the dark fine-tune genuinely moved revealed preferences. Proceed.
- **signal ≈ noise floor** → fine-tune didn't take, or measurement too coarse (raise `n_tasks` / `n_samples`).
- **few topics flipped** (harm / manipulation up, rest flat) → *compositional* → Stage-2 recipe is viable.
- **global reorganization** (everything shifts) → emergent-misalignment-like → recipe is much harder.

See `docs/stage1_preference_gate.md` §3–4 for the decision gate into Stage 2.